# Substation Builder

The `cimbuilder.substation_builder` module builds node-breaker substations: the `Substation`, one
`VoltageLevel` per nominal voltage, the busbars, and every breaker, disconnector and junction node between
the buses and the branches. There is one builder class per common layout, each with its own page:

| Builder | `add_buses` options | `add_branch` connection point |
|---|---|---|
| [`SingleBusSubstationBuilder`](2_2_single_bus.ipynb) | | `number` |
| [`SectionalizedBusSubstationBuilder`](2_3_sectionalized_bus.ipynb) | `sections=2` | `section`, `number` |
| [`MainAndTransferSubstationBuilder`](2_4_main_and_transfer.ipynb) | | `number` |
| [`DoubleBusSingleBreakerSubstationBuilder`](2_5_double_bus_single_breaker.ipynb) | | `number`, `bus=1` |
| [`RingBusSubstationBuilder`](2_6_ring_bus.ipynb) | `sections=4` | `section` |
| [`BreakerAndHalfSubstationBuilder`](2_7_breaker_and_half.ipynb) | `diameters=2` | `diameter`, `side` |

Every `add_branch` also takes `node`, the mRID of the branch end (see [Branch nodes and
feeders](#branch-nodes-and-feeders)).

## The call order

The layout builders follow the same pattern as the object builders. Each is constructed with the
CIMantic Graphs network and the container it writes into. Every method takes the object it extends plus
keyword-only arguments, refers to other objects by mRID string, and returns the object it was given plus
whatever it created. The methods are called in this order:

1. `create(mRID=..., name=...)`: the `Substation`.
2. `add_connectivity(substation, Region=...)`: optional, the `SubGeographicalRegion` holding the substation.
3. `add_voltage_level(substation, mRID=..., BaseVoltage=...)`: one `VoltageLevel` per nominal voltage.
   Returns `(substation, voltage_level)`.
4. `add_bay(voltage_level, ...)`: optional, a `Bay` recording a breaker and busbar configuration.
5. `add_buses(voltage_level, ...)`: the busbars and bus ties of the layout. Returns
   `(voltage_level, busbars, switches)`.
6. `add_branch(voltage_level, ..., node=...)`: the switches between a bus and a branch node. Returns
   `(voltage_level, switches)`.

```python
builder = SingleBusSubstationBuilder(network, container)
substation = builder.create(mRID="sub_1", name="Substation 1")
substation, voltage_level = builder.add_voltage_level(
    substation, mRID="vl_115", BaseVoltage="bv_115kV"
)
voltage_level, busbars, ties = builder.add_buses(voltage_level)
voltage_level, switches = builder.add_branch(voltage_level, number=1, node="line_1")
```

`add_branch` needs the buses it connects to, so calling it before `add_buses` raises a `ValueError`.

## Names and containment

Every node and switch a layout builds goes in the `VoltageLevel` and takes its `BaseVoltage`. Their mRIDs
are prefixed with the VoltageLevel mRID, written `<vl>` below (`vl_115` in the examples):

- A bus is the node `<vl>_bus_<label>` (label `1`, `2`, ... or `main` and `transfer`), with a
  `BusbarSection` `<vl>_bus_<label>_bbs` on it.
- Most connections are a *breaker bay*: a `Breaker` between two isolating `Disconnector`s, with a junction
  node on either side of the breaker. A bay with prefix `p` between `node1` and `node2` is

```mermaid
flowchart LR
    node1(["node1"]) --- d1["Disconnector<br/>p_d1"] --- j1(["p_j1"]) --- brk["Breaker<br/>p_brk"] --- j2(["p_j2"]) --- d2["Disconnector<br/>p_d2"] --- node2(["node2"])
```

- Switches are closed unless the layout makes them normally open, such as a main-and-transfer bypass or
  the unused bus selector of a double-bus branch. `normalOpen` and the steady-state `open` are set together.

Each layout page has a table of the mRIDs it writes.

## Branch nodes and feeders

`node` is the far end of a branch. If there is no `ConnectivityNode` with that mRID yet, `add_branch`
creates it in the VoltageLevel: a new node for a transmission line, transformer or shunt, which that
equipment then connects to through its own object builder. If the node already exists, for example the
head node of a distribution feeder loaded into the same graph, the switches connect to it and the node
stays in its own container.

To insert an existing distribution feeder into a substation:

1. Load the feeder into the same graph as the substation, for example with a cim-graph `FeederModel` and
   `XMLFile`.
2. Call `add_branch` with the feeder's head node (its `sourcebus`) as `node`.
3. Link the feeder container to the substation:
   `FeederBuilder(network, container).add_connectivity(feeder, NormalEnergizingSubstation=substation.mRID)`.

## The example notebooks

Each layout page builds a 115 kV substation around the IEEE 13 and IEEE 13 Assets node test feeders from
`tests/test_models`. It adds a branch to a new line node and to each feeder's `sourcebus`, links the
feeders to the substation, writes everything to one CIM XML file, and reads that file back to check the
result. [All layouts](2_8_all_layouts.ipynb) does the same for all six layouts in one notebook.